# Pneumonia Detection from Chest X-RayTraining script for the custom CNN used in the viewer.Reproduces `ml_models/pneumonia/trained_model.h5`.**Architecture:** 5 Conv+MaxPool blocks (16 → 32 → 64 → 128 → 128 filters) + Flatten + Dense(256) + Dense(512) + Dense(1, sigmoid)**Input:** 300 x 300 x 3 (RGB)**Output:** Binary sigmoid — 0 = Normal, 1 = Pneumonia**Dataset:** Kermany Chest X-Ray — https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia (5,863 images)---

## 1. Setup

In [ ]:
import tensorflow as tffrom tensorflow.keras.models import Sequentialfrom tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Densefrom tensorflow.keras.preprocessing.image import ImageDataGeneratorfrom tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStoppingimport matplotlib.pyplot as pltprint(f'TensorFlow: {tf.__version__}')print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 2. Configuration

In [ ]:
TRAIN_DIR = 'chest_xray/train'VAL_DIR = 'chest_xray/val'TEST_DIR = 'chest_xray/test'IMG_SIZE = (300, 300)INPUT_SHAPE = (300, 300, 3)BATCH_SIZE = 32EPOCHS = 15OUTPUT_MODEL = 'trained_model.h5'

## 3. Data Loading with Augmentation

In [ ]:
train_datagen = ImageDataGenerator(    rescale=1.0/255,    rotation_range=20,    width_shift_range=0.1,    height_shift_range=0.1,    shear_range=0.1,    zoom_range=0.1,    horizontal_flip=True,    fill_mode='nearest',)val_datagen = ImageDataGenerator(rescale=1.0/255)train_gen = train_datagen.flow_from_directory(    TRAIN_DIR,    target_size=IMG_SIZE,    batch_size=BATCH_SIZE,    class_mode='binary',    color_mode='rgb',)val_gen = val_datagen.flow_from_directory(    VAL_DIR,    target_size=IMG_SIZE,    batch_size=BATCH_SIZE,    class_mode='binary',    color_mode='rgb',    shuffle=False,)print(f'Class indices: {train_gen.class_indices}')

## 4. Build Model (Custom CNN)

In [ ]:
model = Sequential([    Conv2D(16, (3, 3), activation='relu', input_shape=INPUT_SHAPE),    MaxPooling2D((2, 2)),    Conv2D(32, (3, 3), activation='relu'),    MaxPooling2D((2, 2)),    Conv2D(64, (3, 3), activation='relu'),    MaxPooling2D((2, 2)),    Conv2D(128, (3, 3), activation='relu'),    MaxPooling2D((2, 2)),    Conv2D(128, (3, 3), activation='relu'),    MaxPooling2D((2, 2)),    Flatten(),    Dense(256, activation='relu'),    Dense(512, activation='relu'),    Dense(1, activation='sigmoid'),])model.compile(    optimizer='adam',    loss='binary_crossentropy',    metrics=['accuracy'],)model.summary()

## 5. Train

In [ ]:
callbacks = [    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7),    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),]history = model.fit(    train_gen,    validation_data=val_gen,    epochs=EPOCHS,    callbacks=callbacks,)

## 6. Save Model

In [ ]:
model.save(OUTPUT_MODEL)print(f'Saved: {OUTPUT_MODEL}')

## 7. Training Curves

In [ ]:
plt.figure(figsize=(12, 4))plt.subplot(1, 2, 1)plt.plot(history.history['loss'], label='Training Loss')plt.plot(history.history['val_loss'], label='Validation Loss')plt.xlabel('Epoch')plt.ylabel('Loss')plt.legend()plt.subplot(1, 2, 2)plt.plot(history.history['accuracy'], label='Training Accuracy')plt.plot(history.history['val_accuracy'], label='Validation Accuracy')plt.xlabel('Epoch')plt.ylabel('Accuracy')plt.legend()plt.tight_layout()plt.show()

## 8. Evaluate on Test Set

In [ ]:
test_gen = val_datagen.flow_from_directory(    TEST_DIR,    target_size=IMG_SIZE,    batch_size=BATCH_SIZE,    class_mode='binary',    color_mode='rgb',    shuffle=False,)results = model.evaluate(test_gen)print(f'Test loss: {results[0]:.4f}')print(f'Test accuracy: {results[1]:.4f}')

## 9. Deploy to ViewerCopy the trained model to the project:```<project-root>/ml_models/pneumonia/trained_model.h5```The Flask viewer loads it automatically on first prediction request.